## Chunking Code Testing

In [1]:
# Testing section-based chunking for fine-tuning
import sys
sys.path.append('../..')
import utils.llm_training as llm_training
from transformers import AutoTokenizer
import re

# Load tokenizer for testing
tokenizer = AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B", trust_remote_code=True)

In [3]:

def chunk_text_by_subsections(text_content: str, tokenizer, max_tokens: int = 2048):
    """
    Helper function to chunk a section by subsections when possible.
    
    Args:
        text_content: The section text to chunk
        tokenizer: The tokenizer to use for counting tokens
        max_tokens: Maximum tokens per chunk
        
    Returns:
        List of text chunks and total token count
    """
    subsection_pattern = r'(\\subsection\{[^}]+\})'
    parts = re.split(subsection_pattern, text_content)
    
    subsections = []
    
    # Handle content before first subsection
    if parts[0].strip():
        subsections.append(parts[0])
    
    # Group subsection headers with their content
    i = 1
    while i < len(parts):
        if re.match(subsection_pattern, parts[i]):  # This is a subsection header
            subsection_content = parts[i]  # Start with the subsection header
            if i + 1 < len(parts):
                subsection_content += parts[i + 1]  # Add the content after the header
            subsections.append(subsection_content)
            i += 2
        else:
            i += 1
    
    # If we only have one subsection (no actual subsection splits), return original
    if len(subsections) <= 1:
        return [text_content], len(tokenizer(text_content, add_special_tokens=False, truncation=False)["input_ids"])
    
    chunks = []
    total_tokens = 0
    current_chunk = ""
    
    for subsection in subsections:
        if not subsection.strip():
            continue
            
        # Check if this would exceed max_tokens when added to current chunk
        test_chunk = current_chunk + subsection
        tokens = tokenizer(test_chunk, add_special_tokens=False, truncation=False)["input_ids"]
        token_count = len(tokens)
        
        if token_count <= max_tokens:
            # Add to current chunk
            current_chunk = test_chunk
        else:
            # Save current chunk if it has content
            if current_chunk.strip():
                chunks.append(current_chunk)
                chunk_tokens = tokenizer(current_chunk, add_special_tokens=False, truncation=False)["input_ids"]
                total_tokens += len(chunk_tokens)
            
            # Check if this subsection alone exceeds max_tokens
            subsection_tokens = tokenizer(subsection, add_special_tokens=False, truncation=False)["input_ids"]
            if len(subsection_tokens) > max_tokens:
                print(f"Subsection too large ({len(subsection_tokens)} tokens), using token-based chunking...")
                # Fall back to token-based chunking for this subsection
                subsection_chunks, subsection_token_count = llm_training.chunk_text(subsection, tokenizer, max_tokens)
                chunks.extend(subsection_chunks)
                total_tokens += subsection_token_count
                current_chunk = ""
            else:
                # Start new chunk with this subsection
                current_chunk = subsection
    
    # Add final chunk if it has content
    if current_chunk.strip():
        chunks.append(current_chunk)
        chunk_tokens = tokenizer(current_chunk, add_special_tokens=False, truncation=False)["input_ids"]
        total_tokens += len(chunk_tokens)
    
    return chunks, total_tokens

def chunk_text_by_sections(text_content: str, tokenizer, max_tokens: int = 2048):
    """
    Chunks text by sections, ensuring each chunk doesn't exceed max_tokens.
    If a section is too large, it tries to split by subsections first.
    If no subsections exist or they're still too large, it falls back to token-based chunking.
    
    Args:
        text_content: The full text to chunk
        tokenizer: The tokenizer to use for counting tokens
        max_tokens: Maximum tokens per chunk
        
    Returns:
        List of text chunks and total token count
    """
    # Find all section boundaries and split into sections
    section_pattern = r'(\\section\{[^}]+\})'
    parts = re.split(section_pattern, text_content)
    print(parts)
    # Group content: first part is pre-section content, then pairs of (section_header, section_content)
    sections = []
    
    # Handle content before first section (title, abstract, etc.)
    if parts[0].strip():
        sections.append(parts[0])
    
    # Group section headers with their content
    i = 1
    while i < len(parts):
        if re.match(section_pattern, parts[i]):  # This is a section header
            section_content = parts[i]  # Start with the section header
            if i + 1 < len(parts):
                section_content += parts[i + 1]  # Add the content after the header
            sections.append(section_content)
            i += 2
        else:
            i += 1
    
    chunks = []
    total_tokens = 0
    current_chunk = ""
    
    for section in sections:
        if not section.strip():
            continue
            
        # Check if this would exceed max_tokens when added to current chunk
        test_chunk = current_chunk + section
        tokens = tokenizer(test_chunk, add_special_tokens=False, truncation=False)["input_ids"]
        token_count = len(tokens)
        
        if token_count <= max_tokens:
            # Add to current chunk
            current_chunk = test_chunk
        else:
            # Save current chunk if it has content
            if current_chunk.strip():
                chunks.append(current_chunk)
                chunk_tokens = tokenizer(current_chunk, add_special_tokens=False, truncation=False)["input_ids"]
                total_tokens += len(chunk_tokens)
            
            # Check if this section alone exceeds max_tokens
            section_tokens = tokenizer(section, add_special_tokens=False, truncation=False)["input_ids"]
            if len(section_tokens) > max_tokens:
                print(f"Section too large ({len(section_tokens)} tokens), trying subsection chunking...")
                # Try to split by subsections first
                subsection_chunks, subsection_token_count = chunk_text_by_subsections(section, tokenizer, max_tokens)
                chunks.extend(subsection_chunks)
                total_tokens += subsection_token_count
                current_chunk = ""
            else:
                # Start new chunk with this section
                current_chunk = section
    
    # Add final chunk if it has content
    if current_chunk.strip():
        chunks.append(current_chunk)
        chunk_tokens = tokenizer(current_chunk, add_special_tokens=False, truncation=False)["input_ids"]
        total_tokens += len(chunk_tokens)
    
    return chunks, total_tokens


In [4]:
# Test with the DPO paper
with open('../../data/arxiv/cleaned_DPO.txt', 'r', encoding='utf-8') as f:
    dpo_paper = f.read()

print("Testing section-based chunking...")
section_chunks, section_total_tokens = chunk_text_by_sections(dpo_paper, tokenizer, max_tokens=2048)

print(f"Section-based chunking: {len(section_chunks)} chunks, {section_total_tokens} total tokens")

# Compare with original token-based chunking
token_chunks, token_total_tokens = llm_training.chunk_text(dpo_paper, tokenizer, 2048)
print(f"Token-based chunking: {len(token_chunks)} chunks, {token_total_tokens} total tokens")

print(f"\nDifference: {len(section_chunks) - len(token_chunks)} chunks")

# Let's examine the chunk boundaries
print("\n=== SECTION-BASED CHUNK BOUNDARIES ===")
for i, chunk in enumerate(section_chunks):
    first_line = chunk.strip().split('\n')[0][:100] + "..." if len(chunk.strip()) > 100 else chunk.strip().split('\n')[0]
    tokens = tokenizer(chunk, add_special_tokens=False, truncation=False)["input_ids"]
    print(f"Chunk {i+1}: {len(tokens)} tokens - '{first_line}'")

print("\n=== TOKEN-BASED CHUNK BOUNDARIES (first 3) ===")
for i, chunk in enumerate(token_chunks[:3]):
    first_line = chunk.strip().split('\n')[0][:100] + "..." if len(chunk.strip()) > 100 else chunk.strip().split('\n')[0]
    tokens = tokenizer(chunk, add_special_tokens=False, truncation=False)["input_ids"]
    print(f"Chunk {i+1}: {len(tokens)} tokens - '{first_line}'")


Testing section-based chunking...
['\\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\n\n\\begin{abstract}\nWhile large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.\nExisting methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).\nHowever, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning to maximize this estimated reward without drifting too far from the original model.\nIn this paper we introduce a new parameterization of the reward model in RLHF that enables extractio

In [5]:
# Test the improved function with subsection handling
print("Testing IMPROVED section+subsection-based chunking...")
improved_chunks, improved_total_tokens = chunk_text_by_sections(dpo_paper, tokenizer, max_tokens=2048*3/2)

print(f"Section+subsection chunking: {len(improved_chunks)} chunks, {improved_total_tokens} total tokens")
print(f"Original token-based chunking: {len(token_chunks)} chunks, {token_total_tokens} total tokens")
print(f"Difference: {len(improved_chunks) - len(token_chunks)} chunks")

print("\n=== IMPROVED SECTION+SUBSECTION CHUNK BOUNDARIES ===")
for i, chunk in enumerate(improved_chunks):
    first_line = chunk.strip().split('\n')[0][:100] + "..." if len(chunk.strip()) > 100 else chunk.strip().split('\n')[0]
    tokens = tokenizer(chunk, add_special_tokens=False, truncation=False)["input_ids"]
    print(f"Chunk {i+1}: {len(tokens)} tokens - '{first_line}'")

# Let's check if we can find any subsections in the text to see if they're being handled
print("\n=== CHECKING FOR SUBSECTIONS IN DPO PAPER ===")
subsection_matches = re.findall(r'\\subsection\{[^}]+\}', dpo_paper)
print(f"Found {len(subsection_matches)} subsections:")
for match in subsection_matches[:5]:  # Show first 5
    print(f"  {match}")

Testing IMPROVED section+subsection-based chunking...
['\\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\n\n\\begin{abstract}\nWhile large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.\nExisting methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).\nHowever, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning to maximize this estimated reward without drifting too far from the original model.\nIn this paper we introduce a new parameterization of the reward model in RLHF th

In [6]:
# Test the integration with fine_tune_on_text function
print("Testing integration with fine_tune_on_text...")

# Import the updated function
import importlib
importlib.reload(llm_training)

# Test that the function signature includes the new parameter
import inspect
sig = inspect.signature(llm_training.fine_tune_on_text)
print(f"fine_tune_on_text parameters: {list(sig.parameters.keys())}")

# Check if chunk_by_section parameter exists and has correct default
chunk_by_section_param = sig.parameters.get('chunk_by_section')
if chunk_by_section_param:
    print(f"chunk_by_section parameter found with default: {chunk_by_section_param.default}")
else:
    print("ERROR: chunk_by_section parameter not found!")

# Test the chunking functions directly
print("\n=== Testing chunking functions directly ===")

# Test section-based chunking function
test_chunks, test_tokens = llm_training.chunk_text_by_sections(dpo_paper, tokenizer, max_tokens=2048*3/2)
print(f"Section-based: {len(test_chunks)} chunks, {test_tokens} tokens")

# Test original chunking function  
orig_chunks, orig_tokens = llm_training.chunk_text(dpo_paper, tokenizer, 2048)
print(f"Token-based: {len(orig_chunks)} chunks, {orig_tokens} tokens")

print(f"Functions working correctly: {test_tokens == orig_tokens}")  # Should have same total tokens


Testing integration with fine_tune_on_text...
fine_tune_on_text parameters: ['model', 'tokenizer', 'log', 'text_content', 'train_cfg', 'train', 'tag', 'callbacks', 'chunk_by_section']
chunk_by_section parameter found with default: False

=== Testing chunking functions directly ===
Section-based: 6 chunks, 12013 tokens
Token-based: 6 chunks, 11928 tokens
Functions working correctly: False


In [22]:
# Test the improved function with subsection handling
print("Testing IMPROVED section+subsection-based chunking...")
improved_chunks, improved_total_tokens = llm_training.chunk_text_by_sections(dpo_paper, tokenizer, max_tokens=2048*5/4)

print(f"Section+subsection chunking: {len(improved_chunks)} chunks, {improved_total_tokens} total tokens")
print(f"Original token-based chunking: {len(token_chunks)} chunks, {token_total_tokens} total tokens")
print(f"Difference: {len(improved_chunks) - len(token_chunks)} chunks")

print("\n=== IMPROVED SECTION+SUBSECTION CHUNK BOUNDARIES ===")
for i, chunk in enumerate(improved_chunks):
    first_line = chunk.strip().split('\n\n')[1][:200] + "..." if len(chunk.strip()) > 200 else chunk.strip().split('\n')[0]
    tokens = tokenizer(chunk, add_special_tokens=False, truncation=False)["input_ids"]
    print(f"Chunk {i+1}: {len(tokens)} tokens - '{first_line}'")

# Let's check if we can find any subsections in the text to see if they're being handled
print("\n=== CHECKING FOR SUBSECTIONS IN DPO PAPER ===")
subsection_matches = re.findall(r'\\subsection\{[^}]+\}', dpo_paper)
print(f"Found {len(subsection_matches)} subsections:")
for match in subsection_matches[:5]:  # Show first 5
    print(f"  {match}")

Testing IMPROVED section+subsection-based chunking...
Section+subsection chunking: 6 chunks, 12013 total tokens
Original token-based chunking: 6 chunks, 11928 total tokens
Difference: 0 chunks

=== IMPROVED SECTION+SUBSECTION CHUNK BOUNDARIES ===
Chunk 1: 2238 tokens - '\begin{abstract}
While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the complet...'
Chunk 2: 1200 tokens - '\section{Preliminaries}\label{section:prelims}...'
Chunk 3: 2103 tokens - '\section{Direct Preference Optimization}\label{sec:DPO}...'
Chunk 4: 1872 tokens - '\section{Theoretical Analysis of DPO}
In this section, we give further interpretation of the DPO method, provide theoretical backing, and relate advantages of DPO to issues with actor critic algorithm...'
Chunk 5: 2218 tokens - '\section{Experiments}
In this section, we empirically evaluate DPO's ability to train policies directly from pre

In [31]:
from importlib import reload
reload(llm_training)
# Test the improved function with subsection handling
print("Testing IMPROVED section+subsection-based chunking...")
improved_chunks, improved_total_tokens = llm_training.chunk_text_by_sections_with_overlap(dpo_paper, tokenizer, max_tokens=2048*5/4)

print(f"Section+subsection chunking: {len(improved_chunks)} chunks, {improved_total_tokens} total tokens")
print(f"Original token-based chunking: {len(token_chunks)} chunks, {token_total_tokens} total tokens")
print(f"Difference: {len(improved_chunks) - len(token_chunks)} chunks")

print("\n=== IMPROVED SECTION+SUBSECTION CHUNK BOUNDARIES ===")
for i, chunk in enumerate(improved_chunks):
    first_line = chunk.strip().split('\n\n')[1][:200] + "..." if len(chunk.strip()) > 200 else chunk.strip().split('\n')[0]
    tokens = tokenizer(chunk, add_special_tokens=False, truncation=False)["input_ids"]
    print(f"Chunk {i+1}: {len(tokens)} tokens - '{first_line}'")

# Let's check if we can find any subsections in the text to see if they're being handled
print("\n=== CHECKING FOR SUBSECTIONS IN DPO PAPER ===")
subsection_matches = re.findall(r'\\subsection\{[^}]+\}', dpo_paper)
print(f"Found {len(subsection_matches)} subsections:")
for match in subsection_matches[:5]:  # Show first 5
    print(f"  {match}")

Testing IMPROVED section+subsection-based chunking...
Section+subsection chunking: 7 chunks, 15896 total tokens
Original token-based chunking: 6 chunks, 11928 total tokens
Difference: 1 chunks

=== IMPROVED SECTION+SUBSECTION CHUNK BOUNDARIES ===
Chunk 1: 2238 tokens - '\begin{abstract}
While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the complet...'
Chunk 2: 1657 tokens - ' These methods represent a convergence of two bodies of work: one body of work on training language models with reinforcement learning for a variety of objectives~\citep{Ranzato2015SequenceLT,paulus20...'
Chunk 3: 2741 tokens - ' Framing the problem as a binary classification we have the negative log-likelihood loss:
\begin{equation}\label{eq:reward_model}
    \mathcal{L}_R(r_{\phi}, \mathcal{D}) = -\mathbb{E}_{(x, y_w, y_l)\...'
Chunk 4: 2685 tokens - ' In Section~\ref{sec:theory}, we further d

In [8]:
test_chunks, test_tokens = llm_training.chunk_text_by_sections(dpo_paper, tokenizer, max_tokens=2048)

print(f"\n=== SECTION-BASED CHUNKS ===")
print(f"Total chunks: {len(test_chunks)}, Total tokens: {test_tokens}")

for i, chunk in enumerate(test_chunks):
    first_line = chunk.strip()
    tokens = tokenizer(chunk, add_special_tokens=False, truncation=False)["input_ids"]
    print(f"Chunk {i+1}: {len(tokens)} tokens - '{first_line}'")
    print()



=== SECTION-BASED CHUNKS ===
Total chunks: 8, Total tokens: 12047
Chunk 1: 1334 tokens - '\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract}
While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.
Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning to maximize this estimated reward without drifting too far from the original model.
In this paper we introduce a new parameterization of 

## Examine Probes that are Not Being Learned

In [40]:
import pandas as pd
df = pd.read_csv('/Users/jlee0/Desktop/research/fine-tuning-or-retrieval/results/FT/SingleArxivPaper_1B_Test_Run_200_Epochs/probe_eval_metrics.csv')
df = pd.read_csv('/Users/jlee0/Desktop/research/fine-tuning-or-retrieval/results/FT/ParaphrasedArxivPaper_1B_Full_Finetuning_Test_Run_10_Epochs/probe_eval_metrics.csv')

In [41]:
# Load the knowledge probes data to understand what each probe_index represents
probes_df = pd.read_csv('/Users/jlee0/Desktop/research/fine-tuning-or-retrieval/data/arxiv/DPO_knowledge_probes_v4.csv')

# Join with the probe evaluation metrics
df_with_probes = df.merge(probes_df.reset_index().rename(columns={'index': 'probe_index'}), on='probe_index', how='left')

In [42]:
df_with_probes.columns

Index(['step', 'probe_index', 'section_x', 'log_prob', 'section.1',
       'perplexity', 'section.2', 'hit_accuracy_at_1', 'section.3',
       'hit_accuracy_at_5', 'section.4', 'hit_accuracy_at_10', 'section_y',
       'subsection', 'section_text', 'subsection_text',
       'raw_knowledge_statement', 'target', 'fact', 'probe'],
      dtype='object')

In [43]:
# Find probes that decreased in hit_accuracy_at_1 from start to end
start_metrics = df_with_probes[df_with_probes['step'] == 0][['probe_index', 'hit_accuracy_at_1']].rename(columns={'hit_accuracy_at_1': 'start_accuracy'})
end_metrics = df_with_probes[df_with_probes['step'] == df_with_probes['step'].max()].rename(columns={'hit_accuracy_at_1': 'end_accuracy'})

accuracy_change = start_metrics.merge(end_metrics, on='probe_index')
accuracy_change['accuracy_drop'] = accuracy_change['start_accuracy'] - accuracy_change['end_accuracy']

# Find probes that decreased
decreased_probes = accuracy_change[accuracy_change['accuracy_drop'] > 0].sort_values('accuracy_drop', ascending=False)
print(f"Found {len(decreased_probes)} probes that decreased in accuracy")

# Display the facts and targets for the decreased probes
for idx, row in decreased_probes.head(10).iterrows():
    probe_idx = row['probe_index']
    print(f"\nProbe {probe_idx} (accuracy drop: {row['accuracy_drop']:.3f})")
    print(f"Fact: '{row['fact']}'")
    print(f"Target: '{row['target']}'")
    print("-" * 50)

decreased_probes.head(10)

Found 13 probes that decreased in accuracy

Probe 3 (accuracy drop: 1.000)
Fact: 'Because large-scale unsupervised language models (LMs) are trained in a completely unsupervised manner, it is difficult to achieve precise control of their behavior'
Target: ' behavior'
--------------------------------------------------

Probe 43 (accuracy drop: 1.000)
Fact: 'Compared to supervised learning, the RLHF pipeline for fine-tuning language models is considerably more complex'
Target: ' complex'
--------------------------------------------------

Probe 303 (accuracy drop: 1.000)
Fact: 'Direct Preference Optimization (DPO) reduces the barrier to training more language models from human preferences compared to existing RLHF algorithms, so DPO reduces the barrier to training'
Target: ' training'
--------------------------------------------------

Probe 19 (accuracy drop: 0.500)
Fact: 'According to our experiments, Direct Preference Optimization (DPO) can fine-tune large language models to align wit

,probe_index,start_accuracy,step,section_x,log_prob,section.1,perplexity,section.2,end_accuracy,section.3,...,hit_accuracy_at_10,section_y,subsection,section_text,subsection_text,raw_knowledge_statement,target,fact,probe,accuracy_drop
3,3,1.000000,100,Title/Abstract,-2.968418,Title/Abstract,19.461105,Title/Abstract,0.000000,Title/Abstract,...,1.000000,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,behavior,Because large-scale unsupervised language mode...,Because large-scale unsupervised language mode...,1.000000
43,43,1.000000,100,Introduction,-2.776970,Introduction,16.070261,Introduction,0.000000,Introduction,...,1.000000,Introduction,No Subsection,\nLarge unsupervised language models (LMs) tra...,\nLarge unsupervised language models (LMs) tra...,While RLHF produces models with impressive con...,complex,"Compared to supervised learning, the RLHF pipe...","Compared to supervised learning, the RLHF pipe...",1.000000
303,303,1.000000,100,Discussion,-2.904158,Discussion,18.249872,Discussion,0.000000,Discussion,...,1.000000,Discussion,No Subsection,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","With virtually no tuning of hyperparameters, D...",training,Direct Preference Optimization (DPO) reduces t...,Direct Preference Optimization (DPO) reduces t...,1.000000
19,19,0.500000,100,Title/Abstract,-4.727944,Title/Abstract,10.633104,Title/Abstract,0.000000,Title/Abstract,...,1.000000,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Our experiments show that DPO can fine-tune LM...,existing methods,"According to our experiments, Direct Preferenc...","According to our experiments, Direct Preferenc...",0.500000
58,58,0.750000,100,Related Work,-12.512051,Related Work,22.828566,Related Work,0.250000,Related Work,...,0.750000,Related Work,No Subsection,\n\nSelf-supervised language models of increas...,\n\nSelf-supervised language models of increas...,"However, their performance on downstream tasks...",human-written completions,Fine-tuning self-supervised language models on...,Fine-tuning self-supervised language models on...,0.500000
138,138,0.500000,100,Direct Preference Optimization,-4.543365,Direct Preference Optimization,9.695703,Direct Preference Optimization,0.000000,Direct Preference Optimization,...,1.000000,Direct Preference Optimization,No Subsection,\label{sec:DPO}\n\nMotivated by the challenges...,\label{sec:DPO}\n\nMotivated by the challenges...,"This way, we fit an implicit reward using an a...",implicit reward,Direct Preference Optimization (DPO) fits an i...,Direct Preference Optimization (DPO) fits an i...,0.500000
158,158,0.500000,100,Theoretical Analysis of DPO,-6.069273,Theoretical Analysis of DPO,20.793423,Theoretical Analysis of DPO,0.000000,Theoretical Analysis of DPO,...,1.000000,Theoretical Analysis of DPO,Your Language Model Is Secretly a Reward Model,"\nIn this section, we give further interpretat...",DPO is able to bypass both fitting an explici...,"Under the Plackett-Luce, and in particular the...",preference distribution,In the context of Direct Preference Optimizati...,In the context of Direct Preference Optimizati...,0.500000
304,304,1.000000,100,Discussion,-1.551366,Discussion,2.172075,Discussion,0.500000,Discussion,...,1.000000,Discussion,No Subsection,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","With virtually no tuning of hyperparameters, D...",hyperparameters,Direct Preference Optimization (DPO) performs ...,Direct Preference Optimization (DPO) performs ...,0.500000
178,178,0.666667,100,Theoretical Analysis of DPO,-6.091549,Theoretical Analysis of DPO,7.618020,Theoretical Analysis of DPO,0.333333,Theoretical Analysis of DPO,...,1.000000,Theoretical Analysis of DPO,Instability of A

In [44]:
# Find probes that increased in perplexity from start to end
start_metrics = df_with_probes[df_with_probes['step'] == 0][['probe_index', 'perplexity']].rename(columns={'perplexity': 'start_perplexity'})
end_metrics = df_with_probes[df_with_probes['step'] == df_with_probes['step'].max()].rename(columns={'perplexity': 'end_perplexity'})

perplexity_change = start_metrics.merge(end_metrics, on='probe_index')
perplexity_change['perplexity_increase'] = perplexity_change['end_perplexity'] - perplexity_change['start_perplexity']

# Find probes that increased
increased_probes = perplexity_change[perplexity_change['perplexity_increase'] > 0].sort_values('perplexity_increase', ascending=False)
print(f"Found {len(increased_probes)} probes that increased in perplexity")

# Display the facts and targets for the increased probes
for idx, row in increased_probes.head(10).iterrows():
    probe_idx = row['probe_index']
    print(f"\nProbe {probe_idx} (perplexity increase: {row['perplexity_increase']:.3f})")
    print(f"Fact: '{row['fact']}'")
    print(f"Target: '{row['target']}'")
    print("-" * 50)

increased_probes.head(10)

Found 53 probes that increased in perplexity

Probe 14 (perplexity increase: 399708.911)
Fact: 'RLHF fine-tunes the language model to maximize the estimated reward without drifting too far from the original model, which is known as drifting'
Target: ' drifting'
--------------------------------------------------

Probe 8 (perplexity increase: 11480.365)
Fact: 'Reinforcement learning from human feedback (RLHF) is a complex procedure for fine-tuning large unsupervised language models that aims to align model behavior with human preferences by maximizing an estimated reward without deviating too far from the original model, which makes RLHF complex'
Target: ' complex'
--------------------------------------------------

Probe 67 (perplexity increase: 8418.510)
Fact: 'Recent methods for fine-tuning large language models by optimizing neural network reward functions and using reinforcement learning algorithms represent a convergence'
Target: ' convergence'
------------------------------------

,probe_index,start_perplexity,step,section_x,log_prob,section.1,end_perplexity,section.2,hit_accuracy_at_1,section.3,...,hit_accuracy_at_10,section_y,subsection,section_text,subsection_text,raw_knowledge_statement,target,fact,probe,perplexity_increase
14,14,15803.682617,100,Title/Abstract,-12.937268,Title/Abstract,415512.593750,Title/Abstract,0.0,Title/Abstract,...,0.0,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...",drifting,RLHF fine-tunes the language model to maximize...,RLHF fine-tunes the language model to maximize...,399708.911133
8,8,2585.859863,100,Title/Abstract,-9.551532,Title/Abstract,14066.224609,Title/Abstract,0.0,Title/Abstract,...,0.0,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...",complex,Reinforcement learning from human feedback (RL...,Reinforcement learning from human feedback (RL...,11480.364746
67,67,30945.357422,100,Related Work,-10.580604,Related Work,39363.867188,Related Work,0.0,Related Work,...,0.0,Related Work,No Subsection,\n\nSelf-supervised language models of increas...,\n\nSelf-supervised language models of increas...,These methods represent a convergence of two b...,convergence,Recent methods for fine-tuning large language ...,Recent methods for fine-tuning large language ...,8418.509766
251,251,6718.288574,100,Experiments,-9.478098,Experiments,13070.301758,Experiments,0.0,Experiments,...,0.0,Experiments,How well can DPO optimize the RLHF objective?,"\nIn this section, we empirically evaluate DPO...",\n\n\begin{figure}\n \centering\n \inclu...,We execute multiple training runs for each alg...,beta,"In our RLHF experiments, the hyperparameter co...","In our RLHF experiments, the hyperparameter co...",6352.013184
83,83,1502.343140,100,Preliminaries,-17.271162,Preliminaries,5628.403320,Preliminaries,0.0,Preliminaries,...,0.0,Preliminaries,No Subsection,\label{section:prelims}\n\nWe review the RLHF ...,\label{section:prelims}\n\nWe review the RLHF ...,It usually includes three phases: 1) supervise...,RL optimization,The RLHF pipeline for language model fine-tuni...,The RLHF pipeline for language model fine-tuni...,4126.060181
276,276,3662.167969,100,Experiments,-17.006231,Experiments,4930.105469,Experiments,0.0,Experiments,...,0.0,Experiments,Can DPO scale to real preference datasets?,"\nIn this section, we empirically evaluate DPO...","\n\label{sec:dpo-real-datasets}\nNext, we eval...",As there is no standard SFT model for this tas...,within distribution,Preferred-FT is used to train a reference mode...,Preferred-FT is used to train a reference mode...,1267.937500
167,167,204.159119,100,Theoretical Analysis of DPO,-13.620123,Theoretical Analysis of DPO,906.926514,Theoretical Analysis of DPO,0.0,Theoretical Analysis of DPO,...,0.5,Theoretical Analysis of DPO,Your Language Model Is Secretly a Reward Model,"\nIn this section, we give further interpretat...",DPO is able to bypass both fitting an explici...,\begin{theorem}\label{thm:main}\n Under mil...,reward classes,"Under mild assumptions, the reparameterization...","Under mild assumptions, the reparameterization...",702.767395
16,16,645.540222,100,Title/Abstract,-13.803309,Title/Abstract,993.917969,Title/Abstract,0.5,Title/Abstract,...,0.5,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"The resulting algorithm, which we call \textit...",performant,"Direct Preference Optimization (DPO), a new al...","Direct Preference Optimization (DPO), a new al...",348.377747
9,9,170.644684,100,Title/Abstract,-6.017079,Title/Abstract,410.378113,Title/Abstract,0.0,Title/Abstract,...,0.0,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, R